# CS 412 Machine Learning
## Spring 2026
## Homework 4

---

**Due:** April 27, 11:55 pm  
**Late Accepted Until:** April 29, 11:55 pm

### Student Information

In [ ]:
NAME       = "Your Name Lastname"     # e.g. "Emine Ayşe Sunar"
STUDENT_ID = "Your Student ID"        # e.g. "12345"

print(f"Student : {NAME}")
print(f"ID      : {STUDENT_ID}")

## Instructions

- **Run all cells before submitting.** All outputs must be visible. The notebook will not be re-run during grading. Cells with no output will receive zero points.
- **Submission:** Upload your notebook as `CS412-HW4-FirstnameLastname.ipynb`.
- **Late policy:** Up to 2 days late accepted with a 10-point penalty per day. Submissions within the first hour after the deadline incur only a 5-point penalty.

---

## Overview

In this assignment, you will build and compare a **Multilayer Perceptron (MLP)** and a **Convolutional Neural Network (CNN)** for image classification on a 10-class subset of **Food-101**, a benchmark dataset of 101,000 food images across 101 categories (pizza, sushi, waffles, etc.), originally published at ECCV 2014. We work with 10 classes to keep training times manageable on Colab. The dataset is available on Sucourse as `Food10.zip`.

**Minimum test accuracy required:**
- MLP: **30%**
- CNN: **50%**

> ⚠️ **Important:** Reaching the minimum accuracy threshold is required. If your model does not meet the threshold, you will receive at most half of the points for that part, even if the rest of your implementation is correct.

---

## Grading

- **Part 1**: Numerical Questions: 20 pts
- **Part 2**: MLP Classifier: 25 pts
- **Part 3**: CNN Classifier: 40 pts
- **Part 4**: Comparison & Analysis: 15 pts
- **Total: 100 pts**

## Part 0: Setup and Data Loading

> 📌 **Note:** This section contains setup and data loading code. Run all cells in this section without modifying them.

Run the cell below to install and import all required libraries. Make sure you are using a GPU runtime on Colab (Runtime > Change runtime type > T4 GPU).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import random

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

### 0.1 Data Loading

Download `Food10.zip` from Sucourse and upload it to your Google Drive **without renaming it**. Then run the cells below to mount your Drive and extract the dataset.

The dataset is a 10-class subset of [Food-101](https://data.vision.ee.ethz.ch/cvl/datasets_extra/food-101/) (Bossard et al., ECCV 2014). It contains 750 training and 250 test images per class across 10 food categories: pizza, sushi, waffles, ice cream, chocolate cake, hamburger, hot dog, ramen, donuts, and fried rice.

In [ ]:
# Colab'da çalışıyorsanız Drive'ı mount edin.
# Lokal/Jupyter ortamında bu hücre hata vermeden geçer.
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive/')
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print(f"IN_COLAB = {IN_COLAB}")

In [ ]:
import os

# Bu notebook hem Colab hem de lokal (Mac/Windows/Linux) çalışacak şekilde ayarlandı.
# - Colab: Drive'a yüklediğiniz Food10.zip'ten /content/data altına çıkarır.
# - Lokal: Proje klasörünüzde ./Food10 veya ./data altında `train/` ve `test/` bekler.

def _has_splits(root: str) -> bool:
    return os.path.isdir(os.path.join(root, "train")) and os.path.isdir(os.path.join(root, "test"))

if IN_COLAB:
    zip_path = "/content/drive/My Drive/Food10.zip"
    target_dir = "/content/data"

    if not _has_splits(target_dir):
        print("Extracting dataset...")
        if not os.path.exists(zip_path):
            raise FileNotFoundError(f"Food10.zip not found at: {zip_path}")
        !unzip -q "{zip_path}" -d "{target_dir}"
        print("Extraction complete.")
    else:
        print("Dataset already extracted, skipping.")

    data_dir = target_dir
else:
    # Lokal: dataset klasörü seçenekleri
    candidates = [
        os.path.join(os.getcwd(), "Food10"),
        os.path.join(os.getcwd(), "data"),
        os.path.join(os.getcwd(), "Food10", "Food10"),
    ]
    data_dir = None
    for c in candidates:
        if _has_splits(c):
            data_dir = c
            break
    if data_dir is None:
        raise FileNotFoundError(
            "Dataset bulunamadı. Lokal çalıştırmak için şu klasörlerden birinde `train/` ve `test/` olmalı:\n"
            + "\n".join([f"- {p}" for p in candidates])
        )

print("Using dataset at:", data_dir)
print("Train dir:", os.path.join(data_dir, "train"))
print("Test  dir:", os.path.join(data_dir, "test"))

### 0.2 Preprocessing and Visualization

We resize all images to 64×64, apply normalization, and split the training data into train (80%) and validation (20%) sets.

In [ ]:
IMG_SIZE = 64
SELECTED_CLASSES = [
    "pizza", "sushi", "waffles", "ice_cream", "chocolate_cake",
    "hamburger", "hot_dog", "ramen", "donuts", "fried_rice"
]

class FoodDataset(Dataset):
    def __init__(self, root_dir, split, transform=None):
        self.transform = transform
        self.samples = []
        split_dir = os.path.join(root_dir, split)
        for label, class_name in enumerate(SELECTED_CLASSES):
            class_dir = os.path.join(split_dir, class_name)
            for fname in os.listdir(class_dir):
                if fname.endswith(".jpg"):
                    self.samples.append((os.path.join(class_dir, fname), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

# Compute mean and std from training set only
def compute_mean_std(dataset):
    loader = DataLoader(dataset, batch_size=64, shuffle=False, num_workers=2)
    mean = torch.zeros(3)
    std = torch.zeros(3)
    for images, _ in loader:
        for c in range(3):
            mean[c] += images[:, c, :, :].mean()
            std[c] += images[:, c, :, :].std()
    mean /= len(loader)
    std /= len(loader)
    return mean, std

pre_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
])
temp_dataset = FoodDataset(data_dir, "train", transform=pre_transform)
mean, std = compute_mean_std(temp_dataset)
print(f"Mean: {mean}")
print(f"Std:  {std}")

# Define transforms using computed stats
transform_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

transform_eval = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Create datasets and loaders
full_train_dataset = FoodDataset(data_dir, "train", transform=transform_train)
test_dataset       = FoodDataset(data_dir, "test",  transform=transform_eval)

val_size   = int(0.2 * len(full_train_dataset))
train_size = len(full_train_dataset) - val_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size],
                                          generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=2)

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
# Visualize one sample per class
display_dataset = FoodDataset(data_dir, "train", transform=transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor()
]))

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
shown = {}
idx = 0
while len(shown) < 10:
    img, label = display_dataset[idx]
    if label not in shown:
        shown[label] = img
    idx += 1

for ax, (label, img) in zip(axes.flatten(), sorted(shown.items())):
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.set_title(SELECTED_CLASSES[label], fontsize=9, pad=4)
    ax.axis("off")

plt.suptitle("One sample per class", fontsize=12)
plt.tight_layout(pad=2.5)
plt.show()

## Part 1: Numerical Questions [20 pts]

> 📌 **Note:** The numerical questions in this section are independent of the Food-101 dataset. The input dimensions and configurations given here are hypothetical and are not related to the Food-101 dataset.

Answer the following questions in the answer cells provided. No code is needed for this part.

### Question 1.1 — Output Size [8 pts]

You have an input feature map of spatial size **40×40** with **3 channels**. You apply a convolutional layer with the following configuration:
- Filter size: 5×5
- Number of filters: 16
- Stride: 2
- Padding: 1

**(a)** What is the spatial size (height × width) of the output feature map? Show your calculation using the formula $W_{out} = \lfloor \frac{W + 2P - F}{S} \rfloor + 1$.

**(b)** What is the total number of learnable parameters in this layer, including biases?

**(c)** What would the output spatial size be if you instead used stride 1 and padding 2? Show your work. What is this special case of padding called, and what is its purpose?

**Your answer for 1.1:**

(a) 
\(W_{out} = \left\lfloor \frac{40 + 2\cdot 1 - 5}{2} \right\rfloor + 1 = \left\lfloor \frac{37}{2} \right\rfloor + 1 = 18 + 1 = 19\).  
Dolayısıyla çıktı uzamsal boyutu **19×19**.

(b) 
Her filtrenin parametresi: \(5\times 5\times 3 = 75\) ağırlık + 1 bias = 76.  
Toplam filtre sayısı 16 olduğundan: \(16\times 76 = 1216\) learnable parametre.

(c) 
Stride \(S=1\), padding \(P=2\):  
\(W_{out} = \left\lfloor \frac{40 + 2\cdot 2 - 5}{1} \right\rfloor + 1 = (40+4-5)+1 = 40\).  
Çıktı uzamsal boyutu **40×40**. Bu, giriş-çıkış boyutunu aynı tutan **"same" padding** (boyutu koruma) özel durumudur; amaç kenar bilgisi kaybını azaltıp uzamsal boyutu korumaktır.

### Question 1.2 — Conceptual Questions [12 pts]

**(a)** What is **weight sharing** in a convolutional layer and why does it make CNNs more parameter-efficient than MLPs for image inputs?

**(b)** A CNN is said to be **translation equivariant**. What does this mean? Give a concrete example using the food images in this dataset.

**(c)** A max pooling layer with a 2×2 window and stride 2 is applied to a feature map of size 32×32×64. What is the output size? How many learnable parameters does this pooling layer have, and why?

**Your answer for 1.2:**

(a) 
**Weight sharing**, aynı konvolüsyon çekirdeğinin (filtrenin) görüntünün tüm uzamsal konumlarında tekrar tekrar uygulanmasıdır. Böylece her konum için ayrı ağırlık öğrenmek yerine tek bir filtre seti öğrenilir; bu da parametre sayısını MLP’ye göre dramatik biçimde azaltır ve özellikle görüntülerdeki yerel desenleri daha verimli öğrenmeyi sağlar.

(b) 
**Translation equivariant** demek: giriş görüntüsü kaydırıldığında, ara özellik haritalarındaki aktivasyonlar da benzer şekilde kayar (yani özellikler “aynı kalır” ama konumu değişir). Örnek: bir pizza dilimi görüntü içinde biraz sağa kaydırılırsa, kenar/yuvarlaklık gibi özellikleri yakalayan filtrelerin yüksek aktivasyon bölgeleri de özellik haritasında sağa kayar.

(c) 
2×2 max pooling, stride 2 ile 32×32 uzamsalı yarıya indirir: çıktı **16×16×64** olur. Pooling katmanı **learnable parametre içermez (0 parametre)** çünkü ağırlık/bias öğrenmez; sadece her pencereden maksimumu seçen deterministik bir işlemdir.

## Part 2: MLP Classifier [25 pts]

In this part, you will implement and train a Multilayer Perceptron (MLP) to classify food images. For the MLP, images must be **flattened** into a 1D vector before being fed into the network.

**Requirement:** Your MLP must achieve at least **30% top-1 accuracy** on the test set.

### 2.1 Model Implementation [10 pts]

Implement your MLP by completing the class below. You are free to choose the number of layers and neurons, but your network must satisfy the following:
- At least 2 hidden layers
- ReLU activations
- At least one Dropout layer

The input dimension is 64×64×3 = 12,288.

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim=64 * 64 * 3, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        return self.net(x)

mlp_model = MLP().to(device)
print(mlp_model)
total_params = sum(p.numel() for p in mlp_model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

### 2.2 Training [10 pts]

Train your MLP using the training set and evaluate on the validation set each epoch. You must:
- Use `CrossEntropyLoss`
- Train for at least 15 epochs
- Print training loss and validation accuracy per epoch
- Plot training loss and validation accuracy curves

In [ ]:
def train_model(model, train_loader, val_loader, num_epochs=20, lr=1e-3):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    train_losses, val_accuracies = [], []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        num_batches = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            num_batches += 1

        avg_loss = running_loss / max(1, num_batches)
        train_losses.append(avg_loss)

        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                logits = model(images)
                preds = torch.argmax(logits, dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        val_acc = 100 * correct / max(1, total)
        val_accuracies.append(val_acc)
        print(f"Epoch [{epoch+1}/{num_epochs}]  Loss: {avg_loss:.4f}  Val Acc: {val_acc:.2f}%")

    return train_losses, val_accuracies

mlp_train_losses, mlp_val_accs = train_model(mlp_model, train_loader, val_loader, num_epochs=20, lr=1e-3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(mlp_train_losses)
axes[0].set_title("MLP Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")

axes[1].plot(mlp_val_accs)
axes[1].set_title("MLP Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")

plt.tight_layout()
plt.show()

### 2.3 Test Evaluation [5 pts]

Evaluate your trained MLP on the test set. You must meet the 30% threshold to receive full marks.

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            preds = torch.argmax(logits, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.detach().cpu().numpy().tolist())
            all_labels.extend(labels.detach().cpu().numpy().tolist())

    acc = 100 * correct / max(1, total)
    return acc, all_preds, all_labels

mlp_test_acc, mlp_preds, mlp_labels = evaluate(mlp_model, test_loader)
print(f"MLP Test Accuracy: {mlp_test_acc:.2f}%")
assert mlp_test_acc >= 30.0, f"Accuracy {mlp_test_acc:.2f}% is below the required 30% threshold!"
print("Threshold requirement met.")

## Part 3: CNN Classifier [40 pts]

Now you will implement a CNN. Unlike the MLP, the CNN operates directly on the 2D spatial structure of the image, so do not flatten the input at the beginning.

**Requirement:** Your CNN must achieve at least **50% top-1 accuracy** on the test set.

### 3.1 Model Implementation [15 pts]

Implement your CNN by completing the class below. Your network must satisfy the following:
- At least 3 convolutional layers
- ReLU activations after each conv layer
- At least one max pooling layer
- At least one Dropout layer in the classifier head
- The final layer must output logits for 10 classes

> 💡 **Tip:** If you are struggling to meet the accuracy threshold, consider adding `nn.BatchNorm2d` after your convolutional layers. Batch normalization stabilizes training and often leads to faster convergence and better performance.

In [ ]:
class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 64 -> 32

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 32 -> 16

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),  # 16 -> 8

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4)),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

cnn_model = CNN().to(device)
print(cnn_model)
total_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

### 3.2 Training [15 pts]

Train your CNN using the same `train_model` function from Part 2. Plot the training curves.

In [ ]:
cnn_train_losses, cnn_val_accs = train_model(cnn_model, train_loader, val_loader, num_epochs=20, lr=1e-3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(cnn_train_losses)
axes[0].set_title("CNN Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")

axes[1].plot(cnn_val_accs)
axes[1].set_title("CNN Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")

plt.tight_layout()
plt.show()

### 3.3 Test Evaluation [10 pts]

Evaluate your trained CNN on the test set. You must meet the 50% threshold to receive full marks.

In [ ]:
cnn_test_acc, cnn_preds, cnn_labels = evaluate(cnn_model, test_loader)

In [ ]:
print(f"CNN Test Accuracy: {cnn_test_acc:.2f}%")
assert cnn_test_acc >= 50.0, f"Accuracy {cnn_test_acc:.2f}% is below the required 50% threshold!"
print("Threshold requirement met.")

## Part 4: Comparison and Analysis [15 pts]

### 4.1 Side-by-Side Validation Accuracy Comparison [5 pts]

Plot both models' validation accuracy on the same graph.

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(mlp_val_accs, label="MLP Val Acc")
plt.plot(cnn_val_accs, label="CNN Val Acc")
plt.axhline(30, linestyle="--", linewidth=1, label="MLP Threshold (30%)")
plt.axhline(50, linestyle="--", linewidth=1, label="CNN Threshold (50%)")
plt.title("Validation Accuracy Comparison")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Final MLP Test Accuracy: {mlp_test_acc:.2f}%")
print(f"Final CNN Test Accuracy: {cnn_test_acc:.2f}%")

### 4.2 Discussion [10 pts]

**(a)** If correctly implemented, the CNN should significantly outperform the MLP on this task. Explain why this is expected using the concepts of **locality** and **weight sharing**. Why are these properties particularly useful for food image classification?

**(b)** Look at your training curves for both models. Which model shows signs of overfitting, underfitting, or neither? Suggest one concrete change you could make to improve the weaker model and explain why it would help.

**Your answer for 4.2:**

(a)
CNN’in MLP’den daha iyi performans göstermesi beklenir çünkü CNN’ler **locality (yerellik)** varsayımıyla çalışır: piksellerdeki anlamlı desenler (kenarlar, dokular, şekiller) yerel komşuluklarda oluşur. Konvolüsyon filtreleri küçük yamalar üzerinde bu yerel desenleri yakalar ve derinleştikçe daha yüksek seviyeli özelliklere (ör. pizza üzerindeki pepperoni benekleri, ramen’deki noodle dokusu) dönüşür. Ayrıca **weight sharing** sayesinde aynı filtre tüm konumlarda kullanılır; böylece hem parametre sayısı azalır hem de aynı desenin görüntünün farklı yerlerinde tanınması kolaylaşır. Yemek sınıflandırmada tabak içindeki nesnenin konumu değişse de (ortada/kenarda) benzer dokular ve şekiller korunur; CNN bu yüzden daha uygundur.

(b)
Eğitim eğrilerinde **MLP** çoğunlukla daha zayıf temsil gücü nedeniyle **underfitting** gösterebilir (loss yüksek kalır, val acc düşük ve yavaş artar) ya da çok parametreli ise düzenleme yetersizliğinden **overfitting** gösterebilir (train loss düşerken val acc plato/azalma). CNN tarafında, yeterli augmentasyon/dropout yoksa overfitting görülebilir; ancak genelde daha iyi genelleme yapar.

Zayıf modeli iyileştirmek için somut bir değişiklik: 
- MLP zayıfsa: **daha güçlü veri artırma** (RandomCrop/ColorJitter) veya **daha uzun eğitim + LR scheduler** eklemek; MLP’nin genellemesini iyileştirir ama CNN kadar yerel yapıdan faydalanamaz.
- CNN zayıf/overfit ise: **Dropout oranını artırmak** veya **weight decay (L2)** eklemek; aşırı uyumu azaltır ve validasyon başarımını yükseltebilir.